# Territorial Transformation Index (TTI) — Reproducible Pipeline
**Buri, V. (2026). LuxVerso Research Initiative. CC-BY 4.0**

---

This notebook reproduces the core TTI computation and national analysis for Brazil (2017–2024) as described in the preprint.

**Requirements:**
- Google Earth Engine account (free tier)
- Google Colab (or local Python environment with `earthengine-api`)

**Pipeline:**
1. GEE authentication
2. TTI computation (core method)
3. Pixel-level sampling (N = 10,000)
4. Municipal aggregation (5,506 municipalities)
5. State aggregation (27 federative units)
6. Export to Drive
7. Exploratory analysis of exported datasets

---

> *'What if, instead of classifying what a territory is, it were possible to measure how much it has become unrecognizable relative to itself?'*

---
## Part 1 — Google Earth Engine: TTI Computation

Run cells 1–5 in Google Colab with GEE access.

In [ ]:
# CELL 1 — GEE Authentication
import ee

# Authenticate and initialize with your own GEE project
# Any free-tier GEE account works.
ee.Authenticate()
ee.Initialize(project='YOUR_GEE_PROJECT_ID')  # replace with your project

print('✅ GEE initialized')

In [ ]:
# CELL 2 — TTI Core Computation
#
# TTI(x, t1, t2) = 1 - E(x,t1) · E(x,t2)
# where E vectors are L2-normalized by construction (unit magnitude).
#
# Reference: Buri (2026), Section 4.1

dataset = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

# Temporal endpoints — maximize cumulative signal over available range
emb_2017 = dataset.filterDate('2017-01-01', '2018-01-01').mosaic()
emb_2024 = dataset.filterDate('2024-01-01', '2025-01-01').mosaic()

# Dot product of L2-normalized vectors = cosine similarity
dot = emb_2017.multiply(emb_2024).reduce(ee.Reducer.sum())

# TTI = 1 - cosine_similarity
# (simplified from full formula because embeddings are already L2-normalized)
tti = ee.Image(1).subtract(dot)

print('✅ TTI image defined')
print('   Formula: TTI = 1 - dot(emb_2017, emb_2024)')
print('   (valid because SATELLITE_EMBEDDING vectors are L2-normalized)')

In [ ]:
# CELL 3 — Brazil boundary and administrative units

# Country boundary
brazil = ee.FeatureCollection('FAO/GAUL/2015/level0') \
    .filter(ee.Filter.eq('ADM0_CODE', 37))

# States (level 1)
states = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM0_CODE', 37))

# Municipalities (level 2)
municipalities = ee.FeatureCollection('FAO/GAUL/2015/level2') \
    .filter(ee.Filter.eq('ADM0_CODE', 37))

print(f'✅ Administrative boundaries loaded')
print(f'   States: {states.size().getInfo()}')
print(f'   Municipalities: {municipalities.size().getInfo()}')

In [ ]:
# CELL 4 — Pixel-level sampling (N = 10,000, seed = 42)
#
# Characterizes the empirical distribution of TTI over Brazil.
# Reference: Buri (2026), Section 5

# Clip TTI to Brazil extent
tti_brazil = tti.clip(brazil.geometry())

# Random sample
pixel_sample = tti_brazil.rename('tti').sample(
    region=brazil.geometry(),
    scale=500,          # 500m subsampling (see paper Section 5 for justification)
    numPixels=10000,
    seed=42,
    geometries=True
)

# Export to Drive
task_sample = ee.batch.Export.table.toDrive(
    collection=pixel_sample,
    description='TTI_pixel_sample_brasil',
    folder='TTI_Brazil_2017_2024',
    fileNamePrefix='TTI_pixel_sample_brasil',
    fileFormat='CSV'
)
task_sample.start()
print('✅ Pixel sample export started → Drive/TTI_Brazil_2017_2024/')
print('   (~5 min to complete)')

In [ ]:
# CELL 5 — Municipal and state aggregation
#
# Reference: Buri (2026), Sections 6.1 and 6.2

# Municipal aggregation (2000m scale)
tti_municipal = tti.rename('tti').reduceRegions(
    collection=municipalities,
    reducer=ee.Reducer.mean().combine(
        ee.Reducer.stdDev(), sharedInputs=True
    ).combine(
        ee.Reducer.percentile([50, 90, 99]), sharedInputs=True
    ),
    scale=2000
)

task_municipal = ee.batch.Export.table.toDrive(
    collection=tti_municipal,
    description='TTI_municipal_brasil',
    folder='TTI_Brazil_2017_2024',
    fileNamePrefix='TTI_municipal_brasil',
    fileFormat='CSV'
)
task_municipal.start()
print('✅ Municipal aggregation export started → Drive/TTI_Brazil_2017_2024/')

# State aggregation
tti_states = tti.rename('tti').reduceRegions(
    collection=states,
    reducer=ee.Reducer.mean().combine(
        ee.Reducer.stdDev(), sharedInputs=True
    ).combine(
        ee.Reducer.percentile([50, 90, 99]), sharedInputs=True
    ),
    scale=2000
)

task_states = ee.batch.Export.table.toDrive(
    collection=tti_states,
    description='TTI_estadual_brasil',
    folder='TTI_Brazil_2017_2024',
    fileNamePrefix='TTI_estadual_brasil',
    fileFormat='CSV'
)
task_states.start()
print('✅ State aggregation export started → Drive/TTI_Brazil_2017_2024/')
print('   (~10–20 min total depending on GEE queue)')

In [ ]:
# CELL 6 — Export national TTI raster (GeoTIFF, 2000m)

task_raster = ee.batch.Export.image.toDrive(
    image=tti_brazil.rename('tti'),
    description='TTI_raster_brazil_2000m',
    folder='TTI_Brazil_2017_2024',
    fileNamePrefix='GW_TTI_2017_2024',
    region=brazil.geometry(),
    scale=2000,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)
task_raster.start()
print('✅ Raster export started → Drive/TTI_Brazil_2017_2024/')
print('   Expected size: ~20–30 MB')
print('   Expected time: ~5–10 min')

---
## Part 2 — Exploratory Analysis of Exported Datasets

Run after GEE exports complete. Loads from Drive.

In [ ]:
# CELL 7 — Setup and load data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

# Load datasets (adjust paths if needed)
BASE = '/content/drive/MyDrive/TTI_Brazil_2017_2024'

df_pixel   = pd.read_csv(f'{BASE}/TTI_pixel_sample_brasil.csv')
df_mun     = pd.read_csv(f'{BASE}/TTI_municipal_brasil.csv')
df_states  = pd.read_csv(f'{BASE}/TTI_estadual_brasil.csv')

# Standardize TTI column name
for df in [df_pixel, df_mun, df_states]:
    for col in df.columns:
        if col.lower() in ['tti', 'mean', 'tti_mean']:
            df.rename(columns={col: 'tti'}, inplace=True)
            break

print(f'✅ Data loaded')
print(f'   Pixel sample:    {len(df_pixel):,} rows')
print(f'   Municipalities:  {len(df_mun):,} rows')
print(f'   States:          {len(df_states):,} rows')

In [ ]:
# CELL 8 — Empirical distribution (replicates Section 5 of the paper)

tti_vals = df_pixel['tti'].dropna()

print('=== TTI Empirical Distribution (N=10,000 pixels, Brazil) ===')
print(f'Mean:    {tti_vals.mean():.4f}')
print(f'Median:  {tti_vals.median():.4f}')
print(f'Std:     {tti_vals.std():.4f}')
print(f'Min:     {tti_vals.min():.4f}')
print(f'P75:     {tti_vals.quantile(0.75):.4f}')
print(f'P90:     {tti_vals.quantile(0.90):.4f}')
print(f'P95:     {tti_vals.quantile(0.95):.4f}')
print(f'P99:     {tti_vals.quantile(0.99):.4f}')
print(f'Max:     {tti_vals.max():.4f}')
print(f'P99/P50: {tti_vals.quantile(0.99)/tti_vals.median():.1f}× (heavy-tail indicator)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.hist(tti_vals, bins=60, color='#2c5f8a', alpha=0.8, edgecolor='white')
ax1.axvline(tti_vals.quantile(0.90), color='#e9c46a', lw=2, ls='--', label='P90')
ax1.axvline(tti_vals.quantile(0.99), color='#e63946', lw=2, ls='--', label='P99')
ax1.set_title('TTI Distribution — N=10,000 pixels, Brazil 2017–2024', fontweight='bold')
ax1.set_xlabel('Territorial Transformation Index (TTI)')
ax1.set_ylabel('Frequency')
ax1.legend()

ax2 = axes[1]
ax2.hist(np.log10(tti_vals + 0.001), bins=60, color='#2a9d8f', alpha=0.8, edgecolor='white')
ax2.set_title('TTI Distribution — log10 scale (heavy-tail visualization)', fontweight='bold')
ax2.set_xlabel('log10(TTI + 0.001)')
ax2.set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(f'{BASE}/TTI_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 9 — State ranking (replicates Table 1 of the paper)

# Identify state name column
name_col = next((c for c in df_states.columns 
                 if any(k in c.lower() for k in ['adm1_name','name','estado','state'])), 
                df_states.columns[0])

ranking = df_states[[name_col, 'tti']].dropna().sort_values('tti', ascending=False)
ranking.columns = ['State', 'Mean_TTI']
ranking['Rank'] = range(1, len(ranking)+1)
ranking['Mean_TTI'] = ranking['Mean_TTI'].round(4)

print('=== State Ranking by Mean TTI ===')
print(ranking[['Rank','State','Mean_TTI']].to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(14, 7))
colors_bar = ['#e63946' if i < 3 else '#457b9d' if i < 10 else '#a8dadc' 
               for i in range(len(ranking))]
ax.barh(ranking['State'][::-1], ranking['Mean_TTI'][::-1], 
        color=colors_bar[::-1], alpha=0.85)
ax.axvline(ranking['Mean_TTI'].mean(), color='gray', ls='--', lw=1.5, 
           label=f'National mean: {ranking["Mean_TTI"].mean():.4f}')
ax.set_title('TTI State Ranking — Brazil 2017–2024', fontweight='bold', fontsize=13)
ax.set_xlabel('Mean Territorial Transformation Index (TTI)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{BASE}/TTI_state_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 10 — Top municipalities (replicates Table 2 of the paper)

mun_name_col = next((c for c in df_mun.columns 
                     if any(k in c.lower() for k in ['adm2_name','name','municipio'])),
                    df_mun.columns[0])
state_col = next((c for c in df_mun.columns 
                  if any(k in c.lower() for k in ['adm1_name','state','uf'])),
                 None)

cols = [mun_name_col, 'tti'] + ([state_col] if state_col else [])
top20 = df_mun[cols].dropna().nlargest(20, 'tti').reset_index(drop=True)
top20.index = top20.index + 1

print('=== Top 20 Municipalities by Mean TTI ===')
print(top20.round(4).to_string())

In [ ]:
# CELL 11 — Validation control points (replicates Table in Section 8.1)
#
# Tests the four control points from the paper against the pixel sample.

control_points = [
    {'name': 'Feira de Santana/BA', 'lat': -12.27, 'lon': -38.97, 'expected': 'high (urban expansion)'},
    {'name': 'Barreiras/BA',        'lat': -12.15, 'lon': -44.99, 'expected': 'medium (agricultural frontier)'},
    {'name': 'Salvador/BA',         'lat': -12.97, 'lon': -38.50, 'expected': 'low (consolidated capital)'},
    {'name': 'Itaparica/BA',        'lat': -12.88, 'lon': -38.68, 'expected': 'low (stable island)'},
]

# If pixel sample has coordinates, find nearest pixel to each control point
if '.geo' in df_pixel.columns or ('longitude' in df_pixel.columns and 'latitude' in df_pixel.columns):
    import json
    
    def get_coords(row):
        try:
            if '.geo' in row.index:
                geo = json.loads(row['.geo'])
                return geo['coordinates'][1], geo['coordinates'][0]
            else:
                return row['latitude'], row['longitude']
        except:
            return None, None
    
    lats, lons = zip(*df_pixel.apply(get_coords, axis=1))
    df_pixel['_lat'] = lats
    df_pixel['_lon'] = lons
    
    print('=== Control Point Validation ===')
    for cp in control_points:
        dist = ((df_pixel['_lat'] - cp['lat'])**2 + 
                (df_pixel['_lon'] - cp['lon'])**2)**0.5
        nearest_tti = df_pixel.loc[dist.idxmin(), 'tti']
        print(f"{cp['name']:30s} TTI={nearest_tti:.4f}  ({cp['expected']})")
else:
    print('Note: pixel sample has no coordinate columns — control point check skipped.')
    print('Paper values: Feira de Santana 0.162 | Barreiras 0.042 | Salvador 0.032 | Itaparica 0.028')

In [ ]:
# CELL 12 — São Francisco case study (Section 7 of the paper)

print('São Francisco / Remanso–BA case study')
print('AOI center: (-42.10, -9.62), radius 30 km')
print()
print('Expected statistics from paper:')
print('  Mean TTI:  0.1326  (3.1× national mean)')
print('  Median:    0.0679  (2.2× national median)')
print('  P90:       0.3697  (4.3× national P90)')
print('  Maximum:   0.6898')
print()
print('To compute in GEE:')
print('''
  aoi = ee.Geometry.Point([-42.10, -9.62]).buffer(30000)
  tti_aoi = tti.clip(aoi)
  stats = tti_aoi.reduceRegion(
      reducer=ee.Reducer.mean().combine(
          ee.Reducer.percentile([50,90,99]), sharedInputs=True
      ).combine(ee.Reducer.max(), sharedInputs=True),
      geometry=aoi, scale=500
  )
  print(stats.getInfo())
''')

---
## Citation

```bibtex
@dataset{buri_2026_tti_brazil,
  author    = {Buri, Vinicius},
  title     = {Territorial Transformation Index (TTI): Embedding-based territorial
               change detection — Application to Brazil (2017–2024)},
  year      = {2026},
  publisher = {Zenodo},
  version   = {v0.6},
  doi       = {10.5281/zenodo.XXXXXXX}
}
```

---
**License:** CC-BY 4.0 | **Contact:** viniburilux@gmail.com | **ORCID:** 0009-0000-6006-1516